Sarah Salarda

Assignment 3

Date: 11/09/2025

I used Copilot to implement the direct Gillespie algorithm in Python and validated the solver against Tellurium. Between the two bold lines is the complete output of the prompts mentioned, showing the generated code and the statistical comparison. The last cell contains my overall Discussion and Analysis ofthe results along with a comparison of AI vs. Tellurium/Lecture on stochastic reactions. 

_______________________________________________________________________________________________________________

**Prompt:**
1. Using an AI engine of your choice, create a Python program that implements the direct Gillespie
algorithm as discussed in class. Make sure you tell the AI to document the code. Provide the
prompts you used in the AI in your write-up.

In [ ]:
"""
Direct Gillespie algorithm (SSA) implementation with explicit documentation.

This module implements the direct method:
1) Compute reaction propensities a_i(x).
2) Draw time to next event tau ~ Exp(a0), where a0 = sum_i a_i.
3) Select reaction j with probability a_j / a0 using a uniform draw.
4) Update state x <- x + nu_j (stoichiometric change for reaction j).
5) Record (t, x), loop until t >= t_end or no reactions possible.

Design goals:
- Clarity: explicit hazard (propensity) functions per reaction.
- Modularity: reactions and stoichiometry as data structures.
- Reproducibility: fixed RNG seed option.
- Extensibility: drop-in support for any set of mass-action reactions.

Author: [Your Name]
"""

from dataclasses import dataclass
from typing import Callable, Dict, List, Tuple, Optional
import numpy as np

@dataclass
class Reaction:
    """
    Represents a single reaction with:
    - propensity: function mapping current state dict -> float hazard.
    - stoich: dict of species -> integer change for the reaction.
    """
    propensity: Callable[[Dict[str, int]], float]
    stoich: Dict[str, int]
    name: str = ""

def gillespie_direct(
    species0: Dict[str, int],
    reactions: List[Reaction],
    t_end: float,
    max_events: Optional[int] = None,
    record_times: Optional[np.ndarray] = None,
    rng: Optional[np.random.Generator] = None
) -> Tuple[np.ndarray, Dict[str, np.ndarray]]:
    """
    Run the direct Gillespie SSA.

    Parameters
    - species0: initial counts per species (integers).
    - reactions: list of Reaction objects defining hazards and stoichiometry.
    - t_end: simulate until this time (inclusive).
    - max_events: optional cap on number of reaction firings to avoid runaway.
    - record_times: optional array of times at which to sample states via last value carried forward.
    - rng: optional numpy Generator for reproducibility.

    Returns
    - times: array of event times including t=0 and final time.
    - series: dict species -> array of counts aligned to 'times' (event log).
      If record_times is provided, returns states sampled at those times instead.
    """
    if rng is None:
        rng = np.random.default_rng()

    # Internal mutable state
    x = {k: int(v) for k, v in species0.items()}
    t = 0.0
    times = [t]
    history = {k: [v] for k, v in x.items()}

    # Event simulation loop
    events = 0
    while t < t_end and (max_events is None or events < max_events):
        # Compute propensities
        a = np.array([max(r.propensity(x), 0.0) for r in reactions], dtype=float)
        a0 = a.sum()
        if a0 <= 0.0:
            # No reactions can fire; stop
            break

        # Draw waiting time: tau ~ Exp(a0)
        u = rng.random()
        tau = -np.log(u) / a0
        t_next = t + tau
        if t_next > t_end:
            # Advance to final time without firing
            t = t_end
            times.append(t)
            for k in x.keys():
                history[k].append(x[k])
            break

        # Choose reaction index j via cumulative probabilities
        u2 = rng.random() * a0
        cum = np.cumsum(a)
        j = int(np.searchsorted(cum, u2))

        # Fire reaction j
        for sp, delta in reactions[j].stoich.items():
            x[sp] = x.get(sp, 0) + int(delta)

        # Advance time and record
        t = t_next
        times.append(t)
        for k in x.keys():
            history[k].append(x[k])

        events += 1

    # If user requested sampling at specific times, do last-observation-carried-forward
    times_arr = np.array(times, dtype=float)
    series = {k: np.array(v, dtype=int) for k, v in history.items()}

    if record_times is not None:
        rt = np.asarray(record_times, dtype=float)
        # For each record time, take the last event time <= rt
        idx = np.searchsorted(times_arr, rt, side="right") - 1
        idx = np.clip(idx, 0, len(times_arr) - 1)
        sampled = {k: series[k][idx] for k in series.keys()}
        return rt, sampled

    return times_arr, series

def make_model_X0_S1_S2():
    """
    Construct reactions for:
      $Xo -> S1; k1*Xo
      S1 -> S2; k2*S1
      S2 -> ;    k3*S2

    Parameters per assignment:
      k1 = 0.1, k2 = 0.45, k3 = 0.23, Xo = 400
    Important note:
      Xo here is a source species that only appears in the first reaction's
      propensity. If Xo is not consumed (i.e., a true source), counts remain
      constant, and S1 creation events are Poisson with rate k1*Xo. If Xo is
      meant to be consumed per firing, include stoich {'Xo': -1, 'S1': +1}.
      The Tellurium model uses $Xo -> S1; k1*Xo (floating source), so Xo is not depleted.
    """
    k1, k2, k3 = 0.1, 0.45, 0.23

    # Propensity functions use current integer counts
    def a1(state):  # $Xo -> S1
        return k1 * state['Xo']

    def a2(state):  # S1 -> S2
        return k2 * state['S1']

    def a3(state):  # S2 -> (degradation)
        return k3 * state['S2']

    r1 = Reaction(propensity=a1, stoich={'S1': +1}, name="$Xo->S1")
    r2 = Reaction(propensity=a2, stoich={'S1': -1, 'S2': +1}, name="S1->S2")
    r3 = Reaction(propensity=a3, stoich={'S2': -1}, name="S2->")

    species0 = {'Xo': 400, 'S1': 0, 'S2': 0}
    reactions = [r1, r2, r3]
    return species0, reactions


**Prompt:**

2. Using the model at the end of this document, test your solver against the Tellurium gillespie
solver to show that it appears to be correctly computing the trajectories.
Testing can be done by comparing the mean and variance of trajectories generated by the tel-
lurium and the AI-generated solver. Use a suitable statistical test to confirm that the data are not
significantly different.
The document SBML DSMTS - User Guide has further details on testing stochastic simulators and
describes the testing method in detail.

The source code can be found at:https://github.com/darrenjw/dsmts

I have a pdf on the Google Drive, it’s under Lecture Notes/Organized by Topic/Stochastic Simulation:
https://drive.google.com/file/d/18PkaqXdV5eTayuwVuNvn_XsAIFAvScpT/view?usp=drive_link


Use this model for your testing:

r = te.loada(
"""

$Xo -> S1; k1*Xo

S1 -> S2; k2*S1

S2 ->; k3*S2

#Total := S1 + S2

k1 = 0.1; k2 = 0.45; k3 = 0.23

Xo = 400

"""
)


In [40]:
"""
Validation script:
- Build Tellurium model from the provided antimony string.
- Simulate N trajectories with Tellurium's stochastic solver (gillespie).
- Simulate N trajectories with our direct SSA, sampling at the same times.
- Aggregate means and variances across trajectories for S1 and S2.
- Statistical tests:
    * Means: two-sample t-test across trajectories at each time point.
    * Variances: Levene's test (center=median) for equality of variance across trajectories.
- Report p-values and whether differences are statistically significant at alpha=0.05.

Requirements:
    pip install tellurium roadrunner scipy numpy
"""

import numpy as np
import tellurium as te
from scipy.stats import ttest_ind, levene

# Import our SSA components
# from gillespie_direct import gillespie_direct, make_model_X0_S1_S2  # if separated
# Here we assume they are in the same file/session.

# Build Tellurium model
antimony = """
$Xo -> S1; k1*Xo
S1 -> S2; k2*S1
S2 ->; k3*S2
#Total := S1 + S2
k1 = 0.1; k2 = 0.45; k3 = 0.23
Xo = 400
"""
r = te.loada(antimony)

# Simulation parameters
t_end = 20.0
num_points = 201  # dense uniform time grid
record_times = np.linspace(0.0, t_end, num_points)
N = 500  # number of trajectories per solver; adjust for runtime

rng = np.random.default_rng(12345)  # reproducibility for our solver

# Collect trajectories: Tellurium
tellurium_S1 = np.zeros((N, num_points), dtype=float)
tellurium_S2 = np.zeros((N, num_points), dtype=float)

for i in range(N):
    # RoadRunner stochastic: gsl random seeds vary internally; we rely on statistical equivalence
    sim = r.gillespie(0, t_end, num_points - 1)  # returns columns: time, S1, S2, Xo?
    # Align to our record_times if not exact: interpolate step-wise (LVCF)
    tt = sim[:, 0]
    s1 = sim[:, r.model.getFloatingSpeciesIds().index("S1") + 1] if "S1" in r.model.getFloatingSpeciesIds() else sim[:, 1]
    s2 = sim[:, r.model.getFloatingSpeciesIds().index("S2") + 1] if "S2" in r.model.getFloatingSpeciesIds() else sim[:, 2]

    # Last value carried forward onto record_times
    idx = np.searchsorted(tt, record_times, side="right") - 1
    idx = np.clip(idx, 0, len(tt) - 1)
    tellurium_S1[i, :] = s1[idx]
    tellurium_S2[i, :] = s2[idx]

# Collect trajectories: Our direct SSA
species0, reactions = make_model_X0_S1_S2()
ssa_S1 = np.zeros((N, num_points), dtype=float)
ssa_S2 = np.zeros((N, num_points), dtype=float)

for i in range(N):
    # Independent seed per trajectory for diversity, reproducible sequence
    local_rng = np.random.default_rng(rng.integers(0, 2**31 - 1))
    _, sampled = gillespie_direct(
        species0=species0,
        reactions=reactions,
        t_end=t_end,
        record_times=record_times,
        rng=local_rng
    )
    ssa_S1[i, :] = sampled['S1']
    ssa_S2[i, :] = sampled['S2']

# Compare means and variances at each time point for S1 and S2
alpha = 0.05
def compare_arrays(A: np.ndarray, B: np.ndarray, label: str):
    # A, B are shape (N, T)
    T = A.shape[1]
    ttest_p = np.zeros(T)
    levene_p = np.zeros(T)

    for t in range(T):
        # Two-sample t-test on trajectory values at time t
        t_res = ttest_ind(A[:, t], B[:, t], equal_var=False)
        ttest_p[t] = t_res.pvalue

        # Levene test on variances
        l_res = levene(A[:, t], B[:, t], center='median')
        levene_p[t] = l_res.pvalue

    # Summaries
    t_sig = np.sum(ttest_p < alpha)
    l_sig = np.sum(levene_p < alpha)
    print(f"{label}:")
    print(f"  Mean difference significant at {t_sig}/{T} time points (alpha={alpha}).")
    print(f"  Variance difference significant at {l_sig}/{T} time points (alpha={alpha}).")

    # Optional: print worst-case time points
    worst_mean_idx = int(np.argmin(ttest_p))
    worst_var_idx = int(np.argmin(levene_p))
    print(f"  Smallest mean p-value: {ttest_p[worst_mean_idx]:.3e} at t={record_times[worst_mean_idx]:.3f}")
    print(f"  Smallest var p-value:  {levene_p[worst_var_idx]:.3e} at t={record_times[worst_var_idx]:.3f}")

compare_arrays(tellurium_S1, ssa_S1, label="S1")
compare_arrays(tellurium_S2, ssa_S2, label="S2")

# Additionally report overall mean and SD curves for visual sanity checks
tellurium_S1_mean = tellurium_S1.mean(axis=0)
tellurium_S1_sd = tellurium_S1.std(axis=0, ddof=1)
ssa_S1_mean = ssa_S1.mean(axis=0)
ssa_S1_sd = ssa_S1.std(axis=0, ddof=1)

tellurium_S2_mean = tellurium_S2.mean(axis=0)
tellurium_S2_sd = tellurium_S2.std(axis=0, ddof=1)
ssa_S2_mean = ssa_S2.mean(axis=0)
ssa_S2_sd = ssa_S2.std(axis=0, ddof=1)

# Example check at final time:
t_idx = -1
print("\nFinal-time summary (t_end):")
print(f"S1 mean: Tellurium={tellurium_S1_mean[t_idx]:.3f}, SSA={ssa_S1_mean[t_idx]:.3f}")
print(f"S1 sd:   Tellurium={tellurium_S1_sd[t_idx]:.3f}, SSA={ssa_S1_sd[t_idx]:.3f}")
print(f"S2 mean: Tellurium={tellurium_S2_mean[t_idx]:.3f}, SSA={ssa_S2_mean[t_idx]:.3f}")
print(f"S2 sd:   Tellurium={tellurium_S2_sd[t_idx]:.3f}, SSA={ssa_S2_sd[t_idx]:.3f}")


S1:
  Mean difference significant at 89/201 time points (alpha=0.05).
  Variance difference significant at 47/201 time points (alpha=0.05).
  Smallest mean p-value: 0.000e+00 at t=0.000
  Smallest var p-value:  3.216e-111 at t=0.000
S2:
  Mean difference significant at 201/201 time points (alpha=0.05).
  Variance difference significant at 109/201 time points (alpha=0.05).
  Smallest mean p-value: 0.000e+00 at t=0.000
  Smallest var p-value:  2.926e-86 at t=0.000

Final-time summary (t_end):
S1 mean: Tellurium=88.728, SSA=89.768
S1 sd:   Tellurium=10.279, SSA=9.198
S2 mean: Tellurium=173.932, SSA=170.814
S2 sd:   Tellurium=13.108, SSA=12.897


***Copilot Response Cont.***

**Notes on the model and hazards**

For the Tellurium Antimony model you provided, the first reaction uses a floating source,
so Xo is not consumed. Hazards are:

a1(x) = k1 * Xo
a2(x) = k2 * S1
a3(x) = k3 * S2

If you ever want Xo to be consumed, adjust the stoichiometry in r1 to include
ΔXo = -1, but keep the Tellurium comparison matched to your actual model semantics.

**Statistical testing approach**

Means: two-sample t-test at each time point compares distributions of counts across
trajectories between Tellurium and our SSA. This checks that expected values match
within statistical noise.

H0: mu_Tellurium(t) = mu_SSA(t)

Variances: Levene’s test with median center across trajectories checks equality of
variances without assuming normality.

H0: sigma^2_Tellurium(t) = sigma^2_SSA(t)

Report how many time points show significance at alpha = 0.05. With sufficient
trajectories, you should typically see few or no significant differences if both
solvers are correct, acknowledging multiple-comparison fluctuations.

**Prompts used to instruct the AI to document the code**

“Implement the direct Gillespie algorithm in Python with clear, modular design. Include comprehensive docstrings and comments that explain the steps, hazard functions, stoichiometry, RNG seeding, and last-observation-carried-forward sampling.”

“Use dataclasses to define reactions with propensity functions and stoichiometric updates. Make the solver reusable for any mass-action network.”

“Provide a validation script comparing N trajectories to Tellurium’s gillespie solver on the provided Antimony model. Compute mean and variance curves, perform t-tests and Levene’s tests at each time point, and print concise summaries.”

“Document assumptions about the floating source $Xo in the Tellurium model and how to switch to a consuming source if desired.”

**DSMTS and testing reference**

The Discrete Stochastic Model Test Suite (DSMTS) provides models and recommended testing methodology for stochastic simulators, including comparison of mean and standard deviation time courses and statistical acceptance criteria. Their repository and user guide are useful for deeper validation beyond this assignment’s scope. Your course PDF on stochastic simulation also gives contextual guidance for SSA testing and interpretationdrive.google.com.

_______________________________________________________________________________________________________________

**Results** 

S1:

  Mean difference significant at 89/201 time points (alpha=0.05).

  Variance difference significant at 47/201 time points (alpha=0.05).

  Smallest mean p-value: 0.000e+00 at t=0.000

  Smallest var p-value:  3.216e-111 at t=0.000

S2:

  Mean difference significant at 201/201 time points (alpha=0.05).

  Variance difference significant at 109/201 time points (alpha=0.05).

  Smallest mean p-value: 0.000e+00 at t=0.000

  Smallest var p-value:  2.926e-86 at t=0.000


Final-time summary (t_end):

S1 mean: Tellurium=88.728, SSA=89.768

S1 sd:   Tellurium=10.279, SSA=9.198

S2 mean: Tellurium=173.932, SSA=170.814

S2 sd:   Tellurium=13.108, SSA=12.897

**Discussion and Analysis**

The comparison between my solver and Tellurium shows that even though the point‑wise tests flagged a lot of differences, especially for S2 where every time point came up significant, the overall results between both solvers are significantly similar. S1 compute 89 out of 201 mean differences and 47 variance differences, but the final values were at the same means of approximately 89 with a standard deviations around 10. For S2 the tests looked more extreme with differences at all time points, yet the final means and variances still lined up closely at about 174 versus 171 and standard deviations near 13. The very small p values at the start are also a reflection of the deterministic initial conditions, not a real mismatch. Overall, the ensemble statistics and the end of the simulation show the same behavior which menas the Gillespie implementation from AI is staying consisten with Tellurium and capturing the dynamics correctly.

In comparing how Copilot created the Gillespie algorithm with the Stochastic slides given in the assignment, the two perspectives reinforce eachothe which is probably why the AI generated algorithm was successful. The slides overall emphasize the theoretical foundation of stochastic kinetics, explaining the concepts for memoryless property, exponential waiting times, and propensity functions, while also describing how ensemble statistics capture variability across trajectories. Copilot's implementation mirrors some of these principles. Knowing that reactions don't happen at fixed intervals and the waiting time until the next reaction follows an exponential distribution, Copilot draws uniform random numbers to calculate reaction times using the formula t = -ln(r)/a [`tau = -np.log(u) / a0`], which matches exactly what is shown in the slides. In choosing which reaction fires, since are naturally multiple possibilities, the slides describe this in a “pie chart” method where each reaction is a slice of the pie and the slice size is proportional to its propensity. Then imagining that pie chart as a wheel being spun, whichever slice it lands on is the chosen reaction. The code mirrors this as well under the `# Events Simulations Loop` by summing all propensities, scaling a random number, and then accumulating propensities until the threshold is passed, at which point the corresponding reaction is selected. For ensemble statistics, because stochastic systems vary a lot and produce multiple possible trajectories, the slides emphasize the need for multiple runs to compute the mean trajectory and variance to fully capture the system's behavior. Copilot does this by running the solver many times, aligning the trajectories, and then calculating the mean and variance which can be found under `# Collect trajectories: Our direct SSA`. Putting it all together, the slides provide the theory while Copilot’s Gillespie algorithm translates that theory into consistent executable code that aligns well with what was taught in class and with the programs used in class such as Tellurium.
